# 03 · Datos crudos: ¿quién es quién?
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/03_datos_crudos_nombres.ipynb)

**Problema:** en registros electorales, la misma persona aparece escrita de varias formas (con y sin tildes, con comas, con un nombre de menos). Antes de cualquier modelo hay que saber quién es quién.

**Los nombres de este ejercicio son ficticios**, construidos para reproducir los problemas reales que aparecen en 555.470 nombres de candidaturas.

In [ ]:
# Preparación: trae los datos del repositorio (funciona igual en Google Colab)
import os, urllib.request
REPO = "https://raw.githubusercontent.com/andrezconz/investigacion-aumentada-ml/main/datos/"

def traer(nombre):
    local = os.path.join("..", "datos", nombre)
    if os.path.exists(local):
        return local
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(REPO + nombre, nombre)
    return nombre

In [ ]:
import pandas as pd
df = pd.read_csv(traer("nombres_ejemplo.csv"))
df

## Paso 1 · Normalizar
Mayúsculas, sin tildes, sin signos y sin espacios dobles. Ojo: la Ñ se vuelve N. Es una decisión, y se documenta.

In [ ]:
import unicodedata, re

def normalizar(texto):
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = re.sub(r"[^A-Za-z ]", " ", t).upper()
    return re.sub(r"\s+", " ", t).strip()

df["normalizado"] = df["nombre_original"].apply(normalizar)
print("Escrituras distintas:", df["nombre_original"].nunique(), "→ después de normalizar:", df["normalizado"].nunique())
df[["nombre_original", "normalizado"]]

## Paso 2 · Nombres incompletos
«PEREZ GOMEZ LUIS» podría ser la misma persona que «PEREZ GOMEZ LUIS ALFONSO»: todas sus palabras están dentro del otro nombre.

In [ ]:
nombres = sorted(df["normalizado"].unique())
def contenido(a, b):
    return set(a.split()) < set(b.split())

pares = [(a, b) for a in nombres for b in nombres if contenido(a, b)]
for a, b in pares:
    print(f"{a:28s} ⊂  {b}")

## La trampa: unir en cadena
Si unimos todos los pares sin pensar, «PEREZ GOMEZ LUIS» sirve de puente y junta a **LUIS ALFONSO** con **LUIS CARLOS**, que son personas distintas.

In [ ]:
def agrupar(pares, nombres):
    grupo = {n: n for n in nombres}
    def raiz(n):
        while grupo[n] != n:
            n = grupo[n]
        return n
    for a, b in pares:
        grupo[raiz(a)] = raiz(b)
    out = {}
    for n in nombres:
        out.setdefault(raiz(n), []).append(n)
    return [g for g in out.values() if len(g) > 1]

print("Unión ingenua (en cadena):")
for g in agrupar(pares, nombres):
    print("  ", " | ".join(g))

## Regla conservadora
Solo aceptamos un grupo si **todos** sus nombres son compatibles entre sí (uno contiene al otro). Si no, no se fusiona: mejor no unir que unir mal.

In [ ]:
def compatible(a, b):
    return a == b or contenido(a, b) or contenido(b, a)

aceptados, ambiguos = [], []
for g in agrupar(pares, nombres):
    ok = all(compatible(a, b) for a in g for b in g)
    (aceptados if ok else ambiguos).append(g)

print("Se unen (misma persona):")
for g in aceptados: print("  ", " | ".join(g))
print("\nAmbiguos, se dejan separados para revisión humana:")
for g in ambiguos: print("  ", " | ".join(g))

> **Tip:** limpiar también es investigar. Escribe la regla en tu ficha .md y prueba con casos de control: dos nombres que **sabes** que son la misma persona y dos que **sabes** que no.

---
## Parte 2 · Detectar lo raro con Isolation Forest
**Cómo funciona:** el modelo hace cortes al azar en los datos. Un caso raro queda aislado con pocos cortes; un caso común necesita muchos.

Los datos de esta parte son **simulados** para mostrar la idea: 300 trayectorias de candidatos con tres rasgos.

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(7)
n = 300
tray = pd.DataFrame({
    "elecciones_previas": rng.poisson(2, n),
    "salto_de_nivel": rng.choice([-1, 0, 0, 0, 1], n),     # 1 = pasa de concejo a asamblea, por ejemplo
    "votos_miles": rng.lognormal(1, .6, n).round(1),
})
# cinco trayectorias poco comunes
tray.loc[n-5:, ["elecciones_previas", "salto_de_nivel", "votos_miles"]] = [[9, 3, 180], [0, 3, 95], [12, -2, 40], [1, 2, 60], [10, 0, 150]]

modelo = IsolationForest(random_state=0).fit(tray)
tray["rareza"] = -modelo.score_samples(tray)
tray.sort_values("rareza", ascending=False).head(8)

> **Tip:** el modelo señala lo raro; no dice por qué es raro ni si está mal. Una trayectoria atípica puede ser un error de digitación o una carrera política excepcional: eso lo decide quien investiga.